In [1]:
!pip install -q PyMuPDF

### Library Imports

In [2]:
import os
import fitz
from pathlib import Path

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


# Document Ingestion

### Input Directories

In [14]:
pdf_directory = "/kaggle/input/datasets/shivanshyadav10/pdf-documents-for-rag"
txt_directory = "/kaggle/input/datasets/shivanshyadav10/text-files-for-rag"

### Function to load pdf

In [4]:
def load_pdf(file_path):

    documents = []

    with fitz.open(file_path) as pdf:
        for page_number, page in enumerate(pdf):

            text = page.get_text()

            if not text.strip():
                continue

            documents.append({
                "text": text,
                "metadata": {
                    "source": Path(file_path).name,
                    "page": page_number + 1,
                    "file_type": "pdf"
                }
            })

    return documents    

### Function to load txt files

In [5]:
def load_txt(file_path):

    with open(file_path, "r", encoding="utf-8") as file:
        text = file.read()
    
    if not text.strip():
        return []

    return [{
        "text": text,
        "metadata": {
            "source": Path(file_path).name,
            "file_type": "txt"
        }
    }]

### Converting the files into text

In [16]:
all_documents = []

for pdf_path in Path(pdf_directory).glob("*.pdf"):
    all_documents.extend(load_pdf(pdf_path))

for txt_path in Path(txt_directory).glob("*.txt"):
    all_documents.extend(load_txt(txt_path))

### Inspecting the loaded documents

In [18]:
print("Total document records:", len(all_documents))

pdf_count = sum(
    1 for doc in all_documents
    if doc["metadata"]["file_type"] == "pdf"
)

txt_count = sum(
    1 for doc in all_documents
    if doc["metadata"]["file_type"] == "txt"
)

print("PDF page records:", pdf_count)
print("TXT file records:", txt_count)

Total document records: 95
PDF page records: 92
TXT file records: 3


# Document Chunking

In [8]:
!pip install -q langchain-text-splitters

### Setting up the overlapping splitter

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    length_function=len,
    separators=["\n\n","\n"," ",""]
)

### Creating chunks of the documents

In [26]:
chunked_documents = []

for document in all_documents:

    chunks = text_splitter.split_text(document['text'])

    for index, chunk in enumerate(chunks):

        chunked_documents.append({
            "text": chunk,
            "metadata": {
                **document["metadata"],
                "chunk_index": index
            }
        })

print("Original document records:", len(all_documents))
print("Total chunks generated:", len(chunked_documents))

Original document records: 95
Total chunks generated: 584


### Inspecting the generated chunks

In [45]:
for chunk in chunked_documents[530:533]:

    print("Metadata:", chunk["metadata"])
    print("Character count:", len(chunk["text"]))
    print("Text preview:\n", chunk["text"][:300])
    print("-" * 60)

Metadata: {'source': 'Lord of the rings.txt', 'file_type': 'txt', 'chunk_index': 10}
Character count: 61
Text preview:
 The Two Towers
Further information: The Two Towers § Contents
------------------------------------------------------------
Metadata: {'source': 'Lord of the rings.txt', 'file_type': 'txt', 'chunk_index': 11}
Character count: 997
Text preview:
 A party of Orcs sent by Saruman and Sauron attack the Fellowship. Boromir tries to protect Merry and Pippin from the Orcs, but they kill him and capture the two Hobbits. Aragorn, Gimli and Legolas decide to pursue the Orcs. The Orcs are killed by Riders of Rohan, led by Éomer. The Hobbits escape int
------------------------------------------------------------
Metadata: {'source': 'Lord of the rings.txt', 'file_type': 'txt', 'chunk_index': 12}
Character count: 952
Text preview:
 Treebeard. The Ents destroy Isengard, Saruman's stronghold, and flood it, trapping the wizard in the tower of Orthanc. Gandalf convinces Treebeard to sen

# Document Embeddings

In [46]:
!pip install -q sentence-transformers

### Loading the model

In [48]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Separating the text from the chunks

In [52]:
texts = [chunk["text"] for chunk in chunked_documents]

### Creating embeddings

In [51]:
embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Embeddings shape: (584, 384)


### Inspecting the embeddings

In [54]:
print("Total chunks:", len(chunked_documents))
print("Total embeddings:", len(embeddings))
print("Dimensions per embedding:", embeddings.shape[1])


Total chunks: 584
Total embeddings: 584
Dimensions per embedding: 384


# Vector store

In [55]:
!pip install -q faiss-cpu

In [56]:
import faiss
import numpy as np

### Creating faiss index for the vectors

In [57]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype("float32"))

print("Total vectors stored:", index.ntotal)

Total vectors stored: 584
